In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Ensure plots render inline inside Jupyter
%matplotlib inline 

from market_data.yf_fred_market_data import MarketDataManager
from market_modelling.path_simulation import HybridValuationVARSimulator
from portfolio_models.linear_models import LongSPYWithTreasuryLadders

initial_nav = 3_000_000
yearly_spending = 450_000
years_to_simulate = 10

random_seed = 42

In [ ]:
portfolio_config = {
    "type": "LongSPYWithTreasuryLadders",
    "equity_allocation": 0.8,
    "ladder_allocation": 0.2,
    "yearly_spending":  yearly_spending,
    "dividend_yield": 0.01,
    "average_inflation": 0.034
}

mdm = MarketDataManager(cache_filepath="market_data.parquet")
levels, returns = mdm.get_aligned_real_returns()
simulator = HybridValuationVARSimulator()
simulator.fit(returns, levels)
strategy = LongSPYWithTreasuryLadders.from_json_object(portfolio_config)

sim_paths = simulator.simulate_paths(years_to_simulate * 12, 1, random_seed)

spx_paths = sim_paths["spx_real"]
tbill_paths = sim_paths["tbill_real"]
tnote_paths = sim_paths["tnote_real"]

nav = strategy.run_simulation(
    spx=spx_paths[0, 1:], 
    yield3m=tbill_paths[0, 1:],
    yield5y=tnote_paths[0, 1:],
    initial_nav=initial_nav,
    months = 12 * years_to_simulate,
    full_book=True)
book = strategy.transaction_book()

output = {
    "spx": spx_paths[0, 1:],
    "yield3m": tbill_paths[0, 1:],
    "yield5y": tnote_paths[0, 1:],
    "nav": nav
}
df_path = pd.DataFrame(output)
df_book = pd.DataFrame(book)

df_book

In [ ]:
spx_trajectory = df_path["spx"]
spx_df = pd.Series(spx_trajectory)
ema1_5 = spx_trajectory.ewm(span=2, adjust=False).mean().values
sma4 = spx_trajectory.rolling(window=4, min_periods=1).mean().values
sma9 = spx_trajectory.rolling(window=9, min_periods=1).mean().values

df = pd.DataFrame({
    "Month": range(0, len(spx_trajectory)),
    "SPX": list(spx_trajectory),
    "EMA30": ema1_5,
    "SMA90": sma4,
    "SMA180": sma9,
    "NAV": df_path["nav"]
})
df.set_index("Month", inplace=True)

# 2. Set up the primary plot
fig, ax1 = plt.subplots(figsize=(12, 6))

# Plot the first time series (Left Y-axis)
color_price = "tab:blue"
ax1.set_xlabel("Day", fontsize=12)
ax1.set_ylabel("SPX Price ($)", color=color_price, fontsize=12)
line1 = ax1.plot(df.index, df["SPX"], color=color_price, linewidth=2, label="SPX Price ($)")
line2 = ax1.plot(df.index, df["EMA30"], color="tab:purple", linewidth=2, label="SPX EMA-30")
line3 = ax1.plot(df.index, df["SMA90"], color="tab:green", linewidth=2, label="SPX SMA-90")
line4 = ax1.plot(df.index, df["SMA180"], color="tab:orange", linewidth=2, label="SPX SMA-180")
ax1.tick_params(axis='y', labelcolor=color_price)
ax1.grid(True, linestyle="--", alpha=0.3)

# 3. Create a twin axis sharing the same X-axis
ax2 = ax1.twinx()  

# Plot the second time series (Right Y-axis)
color_volume = "tab:red"
ax2.set_ylabel("Portfolio NAV", color=color_volume, fontsize=12)
line5 = ax2.plot(df.index, df["NAV"], color=color_volume, linewidth=2, label="NAV")
ax2.tick_params(axis='y', labelcolor=color_volume)

# 4. Combine legends from both axes into a single box
lines = line1 + line2 + line3 + line4 + line5
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc="upper left")

# Title and formatting
plt.title("Simulated SPX vs Portfolio", fontsize=14, fontweight="bold")
fig.autofmt_xdate() # Auto-rotate date labels
plt.tight_layout()

plt.show()

In [ ]:
# 2. Set up the primary plot
fig, ax1 = plt.subplots(figsize=(12, 6))

# Plot the first time series (Left Y-axis)
color_price = "tab:blue"
ax1.set_xlabel("Day", fontsize=12)
ax1.set_ylabel("Yields", color=color_price, fontsize=12)
line1 = ax1.plot(df.index, df_path["yield3m"] * 100.0, color="tab:green", linewidth=2, label="3M T-Bill")
line2 = ax1.plot(df.index, df_path["yield5y"] * 100.0, color="tab:orange", linewidth=2, label="5Y T-Note")
ax1.tick_params(axis='y', labelcolor=color_price)
ax1.grid(True, linestyle="--", alpha=0.3)

lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc="upper left")

# Title and formatting
plt.title("Simulated Interest Rates", fontsize=14, fontweight="bold")
fig.autofmt_xdate() # Auto-rotate date labels
plt.tight_layout()

plt.show()

In [ ]:
# Massage the trade book to make it more presentable
df_book["symbol"] = df_book["symbol"].fillna("").astype(str)
df_book["price"] = (df_book["price"] * 100.0).astype(int) / 100.0
df_book["price"] = df_book["price"].astype(str).replace({"0.0": ""})
df_book["total"] = (df_book["total"] * 100.0).fillna(0).astype(int) / 100.0
df_book["total"] = df_book["total"].astype(str).replace({"0.0": ""})
df_book["rate"] = (df_book["rate"] * 10000.0).fillna(0).astype(int) / 100.0
df_book["rate"] = df_book["rate"].astype(str).replace({"0.0": ""})
df_book["maturity"] = df_book["maturity"].fillna(-1).astype("Int64").astype(str).replace({"-1": ""})
df_book["size"] = df_book["size"].fillna(-1).astype(int).astype(str).replace({"-1": ""})

In [ ]:
# Tell pandas to show all rows
pd.set_option('display.max_rows', None)
# Tell pandas to show all columns (just in case they are hidden too)
pd.set_option('display.max_columns', None)

df_book